# ___Mining and Heavy Equipment___

___Predicting Fuel Consumption for Heavy Duty Applications___

In [ ]:
# Importing nessarcy Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import (
    r2_score, # Coefficeient of determination (model fit)
    mean_squared_error, # penalizes large errors
    mean_absolute_error, # avearage absolue error
    explained_variance_score, # How much variance the model explains
    root_mean_squared_error # Root mean squared error regression loss.
)

## ___1 - Loading Data___

We will try to understand the data, clean, normalize and impute the data where ever possible to derive the best possible result out of the exsisting dataset.

Derived from - https://figshare.com/s/7e6f90fc08adc113f7a7?utm_source=copilot.com&file=54669416

In [ ]:
# Reading Excel
df_main = pd.read_excel(r"D:\ML & AI\_Projects\Mining & Heavy Equipment\final_ori_data.xlsx", index_col = 0).iloc[1:].reset_index(drop=True)
df_main.head()

### ___Adding more data to make it scalable___
Instead of using only vehicle model name, using vehicle specifications as features.

In [ ]:
vehical_specs = {
    "MT96L" : {
        "Engine power (HP)": 523,
        "Engine size (L)": 12.54,
        "Vehicle weight (kg)": 33000,
        "Transmission type": "Automatic",
        "Fuel Type": "Diesel",
        "Numbers of cylinder": 6,
        "Load Capacity": 65000
    },
    "RTH136":{
        "Engine power (HP)": 764,
        "Engine size (L)": 15.93,
        "Vehicle weight (kg)": 48000,
        "Transmission type": "Automatic",
        "Fuel Type": "Diesel",
        "Numbers of cylinder": 6,
        "Load Capacity": 100000
    },
    "SKT105E":{
        "Engine power (HP)": 523,
        "Engine size (L)": 12.54,
        "Vehicle weight (kg)": 38000,
        "Transmission type": "Automatic",
        "Fuel Type": "Diesel",
        "Numbers of cylinder": 6,
        "Load Capacity": 70000
    }
}

In [ ]:
spec_df = pd.DataFrame.from_dict(vehical_specs, orient ='index')
spec_df

In [ ]:
df_main = df_main.merge(spec_df, left_on='vehicle model', right_index=True, how='left')

In [ ]:
df_main

In [ ]:
df_main.to_excel("Heavy Machinenary DataBase.xlsx", index = False)

In [ ]:
print(df_main.dtypes)

In [ ]:
df_main.drop(columns = ['vehicle model'], inplace = True)

In [ ]:
num_cols = df_main.columns

for col in num_cols:
    df_main[col] = pd.to_numeric(df_main[col], errors='ignore')

# then convert numeric columns to float64
num_cols = df_main.select_dtypes(include='number').columns
df_main[num_cols] = df_main[num_cols].astype('float64')

In [ ]:
# verify
print(df_main.dtypes)

In [ ]:
df_main['Actual fueling quantity L'] = pd.to_numeric(
    df_main['Actual fueling quantity L']
    .astype(str)
    .str.replace(r'[^0-9.]', '', regex = True),
    errors = 'coerce'
)
# verify
print(df_main.dtypes)

## ___2 - Basic EDA (Understanding the data)___

* Lets understand the data with the help of ___shape, size, info and describe___
* Understand the ___null count___ present in the data

In [ ]:
df_main.size

In [ ]:
df_main.shape

In [ ]:
df_main.info()

In [ ]:
df_main.describe()

In [ ]:
def null_count(df):
    null_df = pd.DataFrame({
    'Null Count': df.isnull().sum(),
    'Null %': np.round(((df.isnull().sum()/len(df))*100),2)
    })

    null_df = null_df[null_df['Null Count']>0].sort_values(by = "Null %", ascending = False)
    
    return null_df

In [ ]:
null_count(df_main)

## ___3 - Deep EDA (Before Scaling)___

* ___Plotting Graphs___ and having a ___in-depth information___ about the data

In [ ]:
# Distribute the Set of Columns
date_cols = df_main.select_dtypes(include='datetime64[ns]').columns
cat_cols = df_main.select_dtypes(include='string').columns
num_cols = df_main.select_dtypes(include='number').columns

#### ___A) Understanding Numerical Columns___

In [ ]:
stats = df_main[num_cols].describe().T
stats['skew'] = df_main[num_cols].skew()
stats['kurtosis'] = df_main[num_cols].kurt()
stats

In [ ]:
# Plotting Hist Plot for understanding data distribution
for col in num_cols:
    plt.figure(figsize=(10,1))
    sns.histplot(df_main[col], kde=True)
    plt.title(f"Distribution of {col}")
    plt.show()

In [ ]:
# Plotting Box plot to understand the distribution and the outliers
for col in num_cols:
    plt.figure(figsize=(8,1))
    sns.boxplot(x = df_main[col])
    plt.title(f"Boxpolt of {col}")
    plt.show()

In [ ]:
# Plotting the Correlation graph the understang Colinear relation
corr = df_main[num_cols].corr()

plt.figure(figsize = (9,9))
sns.heatmap(corr, annot = True, cmap = 'coolwarm')
plt.title("Correlation Heatmap")
plt.show()

#### ___B) Understanding Categorical Columns___

In [ ]:
for col in cat_cols:
    print(df_main[col].value_counts())

In [ ]:
for col in cat_cols:
    plt.figure()
    sns.countplot(x=df_main[col])
    plt.title(f"Countplot of {col}")
    plt.show()

 ___Target vs Category___

In [ ]:
sns.boxplot(x='shift work',y='Ton-kilometer fuel consumption L / km / t', data=df_main)
plt.show()

In [ ]:
target = 'Ton-kilometer fuel consumption L / km / t'

corr_target = df_main[num_cols].corr()[target].sort_values()

corr_target.plot(kind='barh')
plt.title("Feature importance correlation")
plt.show()


## ___4 - Feature Engineering___

* Deriving new time features from exsisting date
* Dropping columns
* One - Hot encoding categorical variables

In [ ]:
df_main.columns

In [ ]:
# Extracting Usefull data from date 
df_main['dayofyear'] = df_main['date'].dt.dayofyear

In [ ]:
# Cyclic Encoding
df_main['dayofyear_sin'] = np.sin(2*np.pi*df_main['dayofyear']/365)
df_main['dayofyear_cos'] = np.cos(2*np.pi*df_main['dayofyear']/365)

df_main.drop(columns = ['dayofyear'], inplace = True)

In [ ]:
df_main.columns

In [ ]:
# Dropping Date and As we have average temperature we dont need Min and Max Temp
df_main.drop(columns =['date', 'Maximum temperature ( ° F )', 'Minimum temperature ( ° F )'], inplace = True)

In [ ]:
# Only 1 unique value → no variance → dropping them.
df_main.drop(columns = ['Width of working flat', 'Numbers of cylinder'], inplace = True)

In [ ]:
df_main.columns

### ___A) One Hot Encoding for Categorical Varialbles___

In [ ]:
df_main.select_dtypes(include = 'object').columns

In [ ]:
# Categorical Columns 
cat_cols = df_main.select_dtypes(include = 'object').columns

# Encoding
df_main = pd.get_dummies(df_main, columns = cat_cols, drop_first = True)

In [ ]:
df_main.columns

In [ ]:
bool_cols = df_main.select_dtypes(include = 'bool').columns

df_main[bool_cols] = df_main[bool_cols].astype(int)

In [ ]:
df_main.head()

In [ ]:
df_main.nunique()

## ___4 -Splitting Data set in X and Y for Train and Test___

In [ ]:
y = df_main['Ton-kilometer fuel consumption L / km / t']
X = df_main.drop(columns = 'Ton-kilometer fuel consumption L / km / t')

In [ ]:
# Splitting the data Set to avoid data leakage including Train, Validation and test

# Slipt into Train and Temp
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42
)
print("X_train Size:",X_train.shape)
print("X_test Size:",X_test.shape)

Now we have,
* X_train, y_train = 80%
* X_val, y_val = 10%
* X_test, y_test = 10%

## ___5 - Imputing missing values___

### ___Using Iterative Imputer for Numerical Columns___

In [ ]:
print(null_count(df_main))
print(null_count(X_train))
print(null_count(X_test))

In [ ]:
X_train_imp = X_train.copy() # Creating Copy of Train dataframe
X_test_imp = X_test.copy() # Creating Copy of Test dataframe

# Store dataframes in a dictionary
datasets = {
    "train": X_train_imp,
    "test": X_test_imp
}

num_cols = X_train_imp.select_dtypes(include=['number']).columns.tolist()

imputer = IterativeImputer(random_state = 42)

imputer.fit(X_train_imp[num_cols])

for name, data in datasets.items():
    datasets[name][num_cols] = imputer.transform(data[num_cols])

In [ ]:
print(null_count(df_main))
print(null_count(X_train_imp))
print(null_count(X_test_imp))

## ___6 - Scaling Data by using RobustScaler___

In [ ]:
X_train_imp.columns

In [ ]:
X_train_imp.nunique()

In [ ]:
scale_cols = [
    'Mileage km',
    'Actual fueling quantity L',
    'Average temperature ( ° F )',
    'Average wind speed ( knots )',
    'Vehicle weight (kg)',
    'Load Capacity',
]

scaler = RobustScaler()

scaler.fit(X_train_imp[scale_cols])

X_train_imp[scale_cols] = scaler.transform(X_train_imp[scale_cols])
X_test_imp[scale_cols] = scaler.transform(X_test_imp[scale_cols])

In [ ]:
X_train_imp[scale_cols].describe().T

In [ ]:
X_test_imp[scale_cols].hist(figsize=(12,8))
plt.show()

## ___7 - Evaluation Fuction___

In [ ]:
def evaluate_model_df(model, X_train, y_train, X_test, y_test, model_name = 'Model'):

    # Prediction
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Metrics Dictionary
    metrics = {
        "Model": model_name,

        "Train_MSE": mean_squared_error(y_train, y_train_pred),
        "Test_MSE": mean_squared_error(y_test, y_test_pred),
        "Train_RMSE": np.sqrt(mean_squared_error(y_train, y_train_pred)),
        "Test_RMSE": np.sqrt(mean_squared_error(y_test, y_test_pred)),
        "Train_MAE": mean_absolute_error(y_train, y_train_pred),
        "Test_MAE": mean_absolute_error(y_test, y_test_pred),
        "Train_R2": r2_score(y_train, y_train_pred),
        "Test_R2": r2_score(y_test, y_test_pred),
        "Train_Explained_Var": explained_variance_score(y_train, y_train_pred),
        "Test_Explained_Var": explained_variance_score(y_test, y_test_pred)
    }

    return pd.DataFrame([metrics])

In [ ]:
def plot_residual(model, X_test, y_test):
    y_test_pred = model.predict(X_test)
    residuals = y_test - y_test_pred
    
    plt.scatter(y_test_pred, residuals)
    plt.axhline(0, linestyle='--')
    plt.xlabel("Predicted")
    plt.ylabel("Residuals")
    
    return plt.show()

## ___7 - Traning Model Ridge Regression___

### ___A) - Model Ridge Regression___

In [ ]:
from sklearn.linear_model import RidgeCV

alphas = np.logspace(-3, 3, 100)

ridge_cv = RidgeCV(alphas = alphas, cv = 10)
ridge_cv.fit(X_train_imp, y_train)

print("Best alpha:", ridge_cv.alpha_)

In [ ]:
ridge = Ridge(alpha=ridge_cv.alpha_, random_state = 42)

In [ ]:
ridge_scores = cross_val_score(ridge, X_train_imp, y_train, cv = 10, scoring = 'r2')

print("Ridge_CV R2 mean:", ridge_scores.mean())
print("Ridge_CV R2 std:", ridge_scores.std())

In [ ]:
ridge.fit(X_train_imp, y_train)

In [ ]:
ridge_results = evaluate_model_df(
    ridge,
    X_train_imp, y_train,
    X_test_imp, y_test,
    model_name = "Ridge"
)
ridge_results

In [ ]:
plot_residual(ridge, X_test_imp, y_test)

In [ ]:
coefficients = pd.Series(ridge.coef_, index=X_train.columns)
coefficients.sort_values(ascending=False)

### ___B) - Model Random Forst Regression___

In [ ]:
rf = RandomForestRegressor(
    n_estimators = 300,
    max_depth = 8,
    min_samples_split = 5,
    random_state = 42
)

rf_scores = cross_val_score(rf, X_train_imp, y_train, cv = 5, scoring = 'r2')

print("RF CV R2 mean:", rf_scores.mean())
print("RF CV R2 std:", rf_scores.std())

In [ ]:
rf.fit(X_train_imp, y_train)

In [ ]:
rf_results = evaluate_model_df(
    rf,
    X_train_imp, y_train,
    X_test_imp, y_test,
    model_name = "RF"
)
rf_results

In [ ]:
plot_residual(rf, X_test_imp, y_test)

### ___C) - Model XGBoost Regression___

In [ ]:
xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_scores = cross_val_score(xgb, X_train, y_train, cv=5, scoring='r2')

print("XGB CV R2 mean:", xgb_scores.mean())
print("XGB CV R2 std:", xgb_scores.std())

In [ ]:
xgb.fit(X_train_imp, y_train)

In [ ]:
xgb_results = evaluate_model_df(
    xgb,
    X_train_imp, y_train,
    X_test_imp, y_test,
    model_name = "XGB"
)
xgb_results

In [ ]:
plot_residual(xgb, X_test_imp, y_test)

In [ ]:
all_results = pd.concat([ridge_results, rf_results, xgb_results], ignore_index = True)

In [ ]:
all_results